# Homework_2026_03_10 函数-类编程基础



<span style="color:black; font-weight:bold;">请将作业文件命名为 第二次课后练习+姓名+学号.ipynb, 例如 第二次课后练习+张三+1000000000.ipynb</span>

<span style="color:black; font-weight:bold;">在作业过程中觉得有心得或者自己拓展学习到有价值内容的，可以在文件名最后加一个#号。</span>


本次作业覆盖两次课的内容，统一到周日中午提交截止。对类编程内容不熟悉的同学可以先做第一部分，周四课后再完成第二部分。

## section one：函数编程

#### 第零部分：请认真阅读代码，理解代码的功能，先写出预想的结果。运行并检验结果是否如预期。如果不如预期，请分析理解其中的原因

观察以下代码，写出两次`print(c)`的输出结果，并解释原因：

In [4]:
import copy
import sys
a = [1, 2, 3, 4, ['a', 'b']] #原始对象
print(sys.getrefcount(a))
 
b = a                       #赋值，传对象的引用
c = copy.copy(a)            #对象拷贝，浅拷贝
print(sys.getrefcount(a))
d = copy.deepcopy(a)        #对象拷贝，深拷贝
print(sys.getrefcount(a)) 
print(c)

a.append(5)                 #修改对象a
a[4].append('c')            #修改对象a中的['a', 'b']数组对象
 
print( 'a = ', a )
print( 'b = ', b )
print( 'c = ', c )
print( 'd = ', d )

del b
print(sys.getrefcount(a))


2
3
3
[1, 2, 3, 4, ['a', 'b']]
a =  [1, 2, 3, 4, ['a', 'b', 'c'], 5]
b =  [1, 2, 3, 4, ['a', 'b', 'c'], 5]
c =  [1, 2, 3, 4, ['a', 'b', 'c']]
d =  [1, 2, 3, 4, ['a', 'b']]
2


c这里第一次是原来的，第二次列表的列表里面的东西会改变

In [ ]:
src = {'a':{'b':1,'c':2},'d':{'e':3,'f':{'g':4}}}

def flat_dict(dic):
    res={}
    def flat(prefix, dic):
        for key, val in dic.items():
            if type(val)==dict:
                flat(prefix+'.'+key,val) # 递归调用
            else:
                res[prefix+'.'+key]=val
    for key,val in dic.items():
        if type(val)==dict:
            flat(key,val)
        else:
            res[key]=val
    return res

print(flat_dict(src))


{'a.b': 1, 'a.c': 2, 'd.e': 3, 'd.f.g': 4}


这上面的代码把字典的对应关系变成了.,描述了在类中实例的实质。

In [ ]:
# iter()的第二个参数（哨兵模式） 
# 多运行几次观察结果

import random

def read_line():
    """模拟读取数据"""
    return random.choice(['data1', 'data2', 'END', 'data3', 'data4'])

# 创建一个迭代器，直到返回值等于'END'时停止
# 函数也可以是一个迭代器
it = iter(read_line, 'END')

print("开始读取数据，直到遇到'END'：")
for i, data in enumerate(it):
    print(f"第{i+1}次读取: {data}")

开始读取数据，直到遇到'END'：
第1次读取: data1
第2次读取: data1
第3次读取: data1
第4次读取: data1
第5次读取: data3
第6次读取: data2
第7次读取: data1
第8次读取: data2
第9次读取: data1
第10次读取: data2
第11次读取: data3
第12次读取: data3
第13次读取: data3
第14次读取: data3
第15次读取: data2
第16次读取: data4
第17次读取: data1
第18次读取: data4
第19次读取: data3


iter()的第二个参数的主要作用是决定遇到什么停止，默认是StopIteration，即迭代器的元素用完时抛出StopIteration异常。

In [8]:
def environment_scope_demo():
    """
    闭包环境的完整范围
    """
    
    global_var = "全局变量"
    
    def level1(p1, p2):  # 参数
        level1_var = "L1变量"
        
        def level2():
            level2_var = "L2变量"
            
            def level3():
                level3_var = "L3变量"
                
                def closure():
                    # 引用不同层级的变量
                    # - 参数：p1（来自level1）
                    # - 变量：level1_var（来自level1）
                    # - 变量：level2_var（来自level2）
                    # - 变量：level3_var（来自level3）
                    # - 全局变量：global_var（特殊处理）
                    # - 内置函数：len（built-in）
                    
                    return (p1, level1_var, level2_var, 
                           level3_var, global_var, len("test"))
                
                return closure
            
            return level3
        
        return level2
    
    # 创建闭包
    deep_closure = level1("参数1", "参数2")()()
    
    # 分析捕获的环境
    print("=== 环境范围分析 ===")
    
    # 被捕获的自由变量
    free_vars = deep_closure.__code__.co_freevars
    print(f"自由变量名: {free_vars}")
    
    print("\n捕获的值:")
    for name, cell in zip(free_vars, deep_closure.__closure__):
        print(f"  {name}: {cell.cell_contents}")
    
    # 注意：全局变量不在__closure__中
    print(f"\n闭包函数外的变量 'global_var' 不在__closure__中")
    print(f"  通过globals()访问: {deep_closure.__globals__['global_var']}") # error
    
    return deep_closure

environment_scope_demo()

=== 环境范围分析 ===
自由变量名: ('global_var', 'level1_var', 'level2_var', 'level3_var', 'p1')

捕获的值:
  global_var: 全局变量
  level1_var: L1变量
  level2_var: L2变量
  level3_var: L3变量
  p1: 参数1

闭包函数外的变量 'global_var' 不在__closure__中


KeyError: 'global_var'

这是deep_closure中__closure__的实现和__globals__的实现.只有全局变量会进入__globals__中,局部变量不会进入__globals__中。

In [10]:
def calc(operation):
    """
    返回特定运算函数的高阶函数
    
    参数:
        operation: 操作符 '+' 或 '-' 
    
    返回:
        一个接受两个参数的运算函数
    """
    operations = {
        '+': lambda x, y: x + y,
        '-': lambda x, y: x - y,
        '*': lambda x, y: x * y,
        '/': lambda x, y: x / y if y != 0 else "除数不能为零"
    }  # 尝试扩展其他的运算？
    
    return operations.get(operation, lambda x, y: "不支持的操作") # get方法返回一个默认值

# 使用
add = calc('+')
subtract = calc('-')
multiply = calc('*')
orr=calc('--')
print(add(5, 3))      
print(subtract(10, 4)) 
print(multiply(2, 3)) 
print(orr(2, 3))  # 不支持的操作

# 也可以直接调用
print(calc('+')(15, 7))   

8
6
6
不支持的操作
22


In [ ]:
'''functools.wraps 是一个装饰器，它从被装饰的函数复制所有重要属性到包装函数，
   确保装饰后的函数看起来就像原始函数一样，这对于代码的可维护性、调试和文档生成至关重要。
'''
import functools

# 不使用 wraps
def decorator1(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

# 使用 wraps
def decorator2(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator1
def hello1():
    """返回问候语"""
    return "你好"

@decorator2
def hello2():
    """返回问候语"""
    return "你好"

# 直接对比
print(f"不使用wraps - 函数名: {hello1.__name__}")  # wrapper
print(f"不使用wraps - 文档: {hello1.__doc__}")      # None
print(f'不适用wraps - 模块：{hello1.__module__}')  # __main__
print()
print(f"使用wraps - 函数名: {hello2.__name__}")    # hello2
print(f"使用wraps - 文档: {hello2.__doc__}")        # 返回问候语
print(f'使用wraps - 模块：{hello2.__module__}')    # __main__

不使用wraps - 函数名: wrapper
不使用wraps - 文档: None

使用wraps - 函数名: hello2
使用wraps - 文档: 返回问候语


In [ ]:
# 模拟网络连接请求
def decorator_with_params():
    """
    带参数的装饰器（三层嵌套）
    装饰器工厂模式
    """
    
    print("=== 带参数的装饰器 ===\n")
    
    # 基本结构：三层嵌套
    def repeat(times):
        """装饰器工厂：指定重复次数"""
        def decorator(func):
            def wrapper(*args, **kwargs):
                results = []
                for i in range(times):
                    print(f"第{i+1}次调用")
                    result = func(*args, **kwargs)
                    results.append(result)
                return results
            return wrapper
        return decorator
    
    @repeat(3)
    def greet(name):
        return f"Hello, {name}!"
    
    print(f"重复调用结果: {greet('Alice')}")

decorator_with_params()

### 第一部分 代码补全与功能实现

### 1.生成斐波那契数列
    编写一个生成器函数，生成一个无限的斐波那契数列。

In [12]:
def fibonacci():
    # TODO
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
assert [next(fib) for _ in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

### 2.生成器实现两个序列的交错输出
    编写一个生成器函数，接受两个序列作为参数，并生成一个新序列，该序列交错包含原来两个序列的元素。例如，给定序列`[1, 3, 5]`和`[2, 4, 6]`，生成的序列应该是`[1, 2, 3, 4, 5, 6]`。（不要使用`zip`函数）

In [14]:
def interleave(seq1, seq2):
    # TODO
    max_len = max(len(seq1), len(seq2))
    for i in range(max_len):
        if i < len(seq1):
            yield seq1[i]
        if i < len(seq2):
            yield seq2[i]


assert list(interleave([1, 2, 3], [4, 5, 6])) == [1, 4, 2, 5, 3, 6]

### 3.使用递归实现嵌套列表的展平
    编写一个生成器函数，接受一个嵌套列表作为参数，展平该嵌套列表。例如，给定`[1, 2, [3, 4, [5, 6], 7], 8]`，返回`[1, 2, 3, 4, 5, 6, 7, 8]`。
    （提示：可以通过使用isinstance()函数检查元素的类型）

In [15]:
def flatten(lst):
    # TODO
    for item in lst:
        if isinstance(item, list):
            yield from flatten(item)  # 递归调用
        else:
            yield item

nested_list = [1, [2, [3, 4]], 5, [6, 7, [8, 9]]]
assert list(flatten(nested_list)) == [1, 2, 3, 4, 5, 6, 7, 8, 9]

### 4.实现一个简单的装饰器
    编写一个装饰器，为函数添加一个计时功能，打印该函数的运行时间。

In [16]:
import time
def timing_decorator(func):
    # TODO
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"Function {func.__name__} took {execution_time} seconds to execute.")
        return result
    return wrapper

@timing_decorator
def f():
    time.sleep(1)
    
f()

Function f took 1.0012176036834717 seconds to execute.


## Section two 类编程

### 第零部分 阅读理解代码

以下代码的输出是什么？为什么？

In [2]:
class Cat:
    food = 10
    def __init__(self, name):
        self.name = name
    def eat(self):
        Cat.food -= 2

c1 = Cat("Tom")
c2 = Cat("Jerry")
c1.eat()
c2.eat()
print(Cat.food)  # 输出？
print(c1.food)  # 输出？
print(c2.food)  # 输出？

6
6
6


输出6，因为类的属性是实例变量，每个实例都有自己的属性值，所以每个实例的属性值都是6。这个是类的属性，而不是实例变量。

以下代码的输出是什么？为什么？

In [ ]:
class Adder:
    def __call__(self, x, y):
        return x + y

add = Adder()
print(add(3, 5))  # 输出？ 8
class Add:
    def __init__(self, x,y):
        self.x=x
        self.y=y
    def __add__(self, other):
        return self.x+self.y+other
a=Add(3,5)
print(a+7) #输出？15

8


以下代码的输出是什么？(from：geeksforgeeks）

In [4]:
class Mammal():
 
    def __init__(self, name):
        print(name, "Is a mammal")
 
class canFly(Mammal):
 
    def __init__(self, canFly_name):
        print(canFly_name, "cannot fly")
 
        # Calling Parent class
        # Constructor
        super().__init__(canFly_name)
 
class canSwim(Mammal):
 
    def __init__(self, canSwim_name):
 
        print(canSwim_name, "cannot swim")
 
        super().__init__(canSwim_name)
 
class Animal(canFly, canSwim):
 
    def __init__(self, name):
        super().__init__(name)
 
# Driver Code
Carol = Animal("Dog")

Dog cannot fly
Dog cannot swim
Dog Is a mammal


继承是从内往外的，所以输出是... fly,... swim, ... mammal

以下代码的输出是什么？解释原因：

In [ ]:
class Math:
    num=10
    @staticmethod # 静态方法,意思是不需要实例化就可以调用
    def add(x, y):
        return cls.num+x + y #报错
    @classmethod # 类方法,第一个参数固定为类本身
    def mul(cls, x, y):
        return cls.num+x * y

print(Math.add(2, 3))
print(Math.mul(2, 3))
math = Math()
math2 = Math()
print(math.add(2, 3))
print(math.mul(2, 3))
print(math.add is math2.add)
print(math.add is Math.add)
print(math.mul is Math.mul)
print(math.mul is math2.mul)

NameError: name 'cls' is not defined

以下代码的输出是什么？解释 `Memoize` 类的作用：

In [ ]:
class Memoize:
    def __init__(self, func):
        self.func = func
        self.memo = {}
    def __call__(self, x):
        if x not in self.memo:
            self.memo[x] = self.func(x)
        return self.memo[x]

@Memoize
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

print(fib(5))  # 输出？ 5

5


memorize作用是记忆化储存函数的结果，避免重复计算。

以下代码的输出是什么？解释 `__next__` 的逻辑：

In [ ]:
class Counter:
    def __init__(self, limit):
        self.limit = limit
        self.n = 0
    def __iter__(self):
        return self
    def __next__(self):
        if self.n >= self.limit:
            raise StopIteration
        self.n += 1
        return self.n

for num in Counter(3):
    print(num)

以下代码的输出是什么？解释为什么：

In [12]:
class FileSystemNode:
    def __init__(self, name):
        self.name = name
        self.__children = []  
    
    def __str__(self):       
        return f"File[{self.name}]"

    @property
    def children(self):      # 通过属性访问器暴露伪私有变量
        return self.__children

class LoggerDecorator:      
    def __init__(self, func):
        self.func = func
        self.create_count = 0
    
    def __call__(self, *args):  
        self.create_count += 1
        print(f"节点创建事件 {self.create_count}")
        return self.func(*args)

class Directory(FileSystemNode):  
    MAX_CHILDREN = 5  # 目录最多容纳5个子节点
    
    def add_child(self, node):
        if len(self.children) >= self.MAX_CHILDREN:
            raise ValueError("子节点数量超过限制")  
        self.children.append(node)

# 应用类装饰器
@LoggerDecorator
def create_node(name):
    return FileSystemNode(name)

root = create_node("root")          # 触发装饰器打印日志
doc_dir = Directory("我的文档")
for i in range(6):                  # 将触发第6次报错
    try:
        doc_dir.add_child(create_node(f"doc{i}"))
    except ValueError as e:
        print(f"操作异常：{e}")      
        assert "子节点数量超过限制" in str(e), "异常提示错误"


节点创建事件 1
节点创建事件 2
节点创建事件 3
节点创建事件 4
节点创建事件 5
节点创建事件 6
节点创建事件 7
操作异常：子节点数量超过限制


节点创建事件....
...
...输出就是这个



### 第一部分：基础练习

#### 1.1 类的初始化方法
    补全以下类的 `__init__` 方法，使其能正确初始化属性：

In [13]:
class Student:
    def __init__(self, name, score):
        self.name=name
        self.score=score
        # 填入代码：初始化实例属性 name 和 score
        
# 测试
s = Student("Alice", 90)
assert s.name == "Alice" and s.score == 90, "属性初始化错误"

#### 1.2 实现完全独立的副本返回函数
    练习深拷贝逻辑：按提示填写代码跑通后理解就行


In [16]:
import copy

def creating_copies():
    """演示创建独立副本的方法"""
    
    # 方法：使用类（生成独立副本）
    class DataContainer:
        """使用类实现独立副本"""
        
        def __init__(self, initial_data):
            # 【填空2】保存initial_data的深拷贝到self._data
            self._data = copy.deepcopy(initial_data)  # 提示：使用copy.deepcopy()
        
        def get(self):
            # 【填空3】返回数据的副本，保护封装性
            return self._data.copy()  # 提示：返回self._data的副本，而不是直接返回引用
        
        def update(self, new_data):
            # 【填空4】更新内部数据
            self._data = copy.deepcopy(new_data)  # 提示：同样需要深拷贝
        
        def __repr__(self):
            return f"DataContainer({self._data})"
    
    print("\n=== 使用类实现独立副本 ===")
    # 【填空5】创建两个独立的DataContainer实例，都使用[1, 2, 3]作为初始数据
    container1 = DataContainer([1, 2, 3]) # 提示：DataContainer([1, 2, 3])
    container2 = DataContainer([1,2,3])  # 提示：同样参数，应该是独立的副本
    
    print(f"  container1: {container1.get()}")
    print(f"  container2: {container2.get()}")
    
    # 验证独立性：修改一个不影响另一个
    # 【填空6】获取container1的数据并添加元素4
    data1 = container1.get()  # 提示：使用container1.get()
    data1.append(4)
    
    print(f"\n修改container1后:")
    print(f"  container1: {container1.get()}")
    print(f"  container2: {container2.get()}")  # 应该仍然是[1, 2, 3]
creating_copies()


=== 使用类实现独立副本 ===
  container1: [1, 2, 3]
  container2: [1, 2, 3]

修改container1后:
  container1: [1, 2, 3]
  container2: [1, 2, 3]


#### 1.3 继承中的 `super()`
    补全子类的初始化方法：


In [17]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed):
        # 填入代码：调用父类初始化并添加新属性bread
        super().__init__(name)
        self.breed = breed
        

d = Dog("Buddy", "Golden")
assert d.name == "Buddy" and d.breed == "Golden", "继承初始化错误"

#### 1.4 魔法方法 `__str__`
    补全代码，使 `print(p)` 输出 `"Point(3,4)"`：


In [18]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __str__(self):
        # 填入代码：返回格式化字符串
        return "Point({},{})".format(self.x, self.y)

p = Point(3, 4)
print(p)  # 应输出 "Point(3,4)"
assert str(p) == "Point(3,4)", "格式化字符串错误"

Point(3,4)


#### 1.5 类装饰器实现
    补全类装饰器，统计函数调用次数：


In [19]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.calls = 0
    def __call__(self, *args):
        # 填入代码：统计调用次数并执行函数
        self.calls += 1
        return self.func(*args)

@CountCalls
def greet():
    print("Hello")

greet()
greet()
assert greet.calls == 2, "装饰器统计错误"

Hello
Hello


#### 1.6 属性访问控制
    补全 `@property` 装饰器，实现只读属性：


In [ ]:
class Circle:
    def __init__(self, radius):
        self.__radius = radius
    @property
    def radius(self):
        # 填入代码：返回私有属性

c = Circle(5)
print(c.radius)  # 应输出 5
c.radius = 10    # 应报错：只读属性

#### 1.7 树结构实现
    补全二叉树插入左子节点的逻辑：


In [21]:
class BinaryTree:
    def __init__(self, root):
        self.key = root
        self.left = None
        self.right = None
    def insert_left(self, node):
        if self.left is None:
            self.left = BinaryTree(node)  # 填入新节点创建代码
        else:
            new_node = BinaryTree(node)
            new_node.left = self.left
            self.left = new_node

tree = BinaryTree("A")
tree.insert_left("B")
assert tree.left.key == "B", "左子节点插入失败"

### 第二部分：进阶练习，代码补全。要求补全代码并通过验证语句。

#### 2.1 二叉搜索树
    补全代码，使用迭代器类实现对二叉搜索树的中序遍历

In [1]:
import sys
from io import StringIO

class TreeNode:
    '''二叉搜索树节点的定义'''
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

class OperationTree:
    '''二叉搜索树操作'''
    def insert(self, root, val):
        '''二叉搜索树插入操作'''
        if root == None:
            root = TreeNode(val)
        elif val < root.val:
            root.left = self.insert(root.left, val)
        elif val > root.val:
            root.right = self.insert(root.right, val)
        return root

class BSTIterator:
    def __init__(self, root):
        self.nodes = []
        self._inorder_traversal(root)
        self.index = 0

    def _inorder_traversal(self, node):
        # 代码填空：实现中序遍历的逻辑
        if node:
            self._inorder_traversal(node.left)
            self.nodes.append(node.val)
            self._inorder_traversal(node.right)
        else:
            return
        

    def __iter__(self):
        # 代码填空：实现迭代器的逻辑
        return self



    def __next__(self):
        # 代码填空：实现next方法的逻辑
        if self.index >= len(self.nodes):
            raise StopIteration
        val = self.nodes[self.index]
        self.index += 1
        return val

# 测试
List = [17,5,35,2,11,29,38,9,16,8]
root = None
op = OperationTree()
for val in List:
    root = op.insert(root,val)

iterator = BSTIterator(root)
for idx,val in enumerate(iterator):
    print(val)
    assert val == sorted(List)[idx], f"Error: {val} != {sorted(List)[idx]}"


2
5
8
9
11
16
17
29
35
38


## 选做题

下面这段代码实现了一个可hash的学生类，先阅读理解这段代码，尝试以此为基础实现一个选课系统（可以用AI辅助），

要求实现每个学生可以独立选退课程：Courses = [('三宝', 2), ('音数', 2), ('地概', 2), ('高数', 4), ('线代', 3), ('棒垒', 1)]

课程人数上限为3，每人的学分上限为6。超过了在最后在合并信息阶段进行合理的剔除（这里有个最优化问题，有点难）。

可以尝试用AI实现最终课程信息合并后生成每个学生的最终选课结果。重点关注采用什么策略。可以考虑动态（学生有自主性）或静态策略（系统最后自动平衡）才能最有效的利用资源并满足大家的需求。请把算法思路单独用文本总结出来放在前面的注释里。能完成的可以给作为的文件名后加一个# 基础好的同学建议尽早提交



In [ ]:
from typing import Set, Optional, Union, List, Dict
from dataclasses import dataclass, field
from datetime import datetime
import hashlib
import json

class Student:
    """可哈希的学生类，使用学号作为唯一标识"""
    
    def __init__(self, student_id: str, name: str, courses: Optional[Set[str]] = None):
        """
        初始化学生对象
        
        Args:
            student_id: 学号（唯一标识）
            name: 姓名
            courses: 选课集合
        """
        self.student_id = student_id
        self.name = name
        self._courses = set(courses) if courses else set()
    
    @property
    def courses(self) -> Set[str]:
        """返回选课集合的副本，保护封装性"""
        return self._courses.copy()
    
    def add_course(self, course: str) -> None:
        """添加课程"""
        self._courses.add(course)
    
    def remove_course(self, course: str) -> None:
        """移除课程"""
        self._courses.discard(course)  # 使用discard避免KeyError
    
    def __hash__(self) -> int:
        """
        哈希方法：只基于学号计算哈希值
        这样两个学号相同但其他属性不同的对象会被视为同一个键
        """
        return hash(self.student_id)
    
    def __eq__(self, other: object) -> bool:
        """
        相等性比较：只比较学号
        确保哈希一致性：如果 x == y，则 hash(x) == hash(y)
        """
        if not isinstance(other, Student):
            return NotImplemented
        return self.student_id == other.student_id
    
    def __lt__(self, other: 'Student') -> bool:
        """小于比较，用于排序"""
        return self.student_id < other.student_id
    
    def __repr__(self) -> str:
        """开发人员友好的表示"""
        courses_str = f", courses={sorted(self._courses)}" if self._courses else ""
        return f"Student('{self.student_id}', '{self.name}'{courses_str})"
    
    def __str__(self) -> str:
        """用户友好的表示"""
        courses_str = ', '.join(sorted(self._courses)) if self._courses else "无选课"
        return f"学生：{self.name}({self.student_id}) 选课：{courses_str}"
    

# 测试基础功能
def test_basic_student():
    """测试Student类的基础功能"""
    print("=" * 50)
    print("测试基础功能")
    print("=" * 50)
    
    # 创建学生
    s1 = Student("2024001", "张三", {"数学", "物理"})
    s2 = Student("2024002", "李四", {"计算机", "数学"})
    s3 = Student("2024001", "张三", {"英语"})  # 学号相同
    
    print(f"s1: {s1}")
    print(f"s2: {s2}")
    print(f"s3: {s3}")
    
    # 测试哈希和相等性
    print(f"\n哈希值比较:")
    print(f"hash(s1) = {hash(s1)}")
    print(f"hash(s2) = {hash(s2)}")
    print(f"hash(s3) = {hash(s3)}")
    print(f"s1 == s2: {s1 == s2}")
    print(f"s1 == s3: {s1 == s3}")  # 应该为True，因为学号相同
    
    # 测试集合行为
    student_set = {s1, s2, s3}  # s1和s3被视为同一个元素
    print(f"\n学生集合大小: {len(student_set)}")  # 应该是2
    for student in student_set:
        print(f"  {student}")

if __name__ == "__main__":
    test_basic_student()

In [2]:
"""
选课系统优化策略：
采用静态集中分配策略。所有学生先提交选课志愿（可超限），然后系统在最后阶段进行统一调整，
以满足课程人数上限（每门课最多3人）和个人学分上限（每人最多6学分）。
调整算法为贪心分配：首先将所有学生随机打乱顺序以保证公平，然后按此顺序依次处理每个学生。
对于每个学生，遍历其志愿课程列表（按课程学分降序排列，以优先保留高学分课程），尝试将课程
加入该学生的最终选课集合，但需满足两个条件：该课程当前已选人数未达上限，且加入后该学生
总学分不超过上限。若满足则保留，否则跳过。这样一次遍历即可得到满足所有约束的分配结果。
该策略简单高效，能在保证约束的前提下尽量满足学生志愿，且随机顺序避免了顺序偏见。
"""

import random
from typing import Set, Optional, Union, List, Dict
from dataclasses import dataclass, field
from datetime import datetime
import hashlib
import json

# -------------------- 原有的 Student 类（略作扩展，添加清空课程方法）--------------------
class Student:
    """可哈希的学生类，使用学号作为唯一标识"""
    
    def __init__(self, student_id: str, name: str, courses: Optional[Set[str]] = None):
        """
        初始化学生对象
        
        Args:
            student_id: 学号（唯一标识）
            name: 姓名
            courses: 选课集合
        """
        self.student_id = student_id
        self.name = name
        self._courses = set(courses) if courses else set()
    
    @property
    def courses(self) -> Set[str]:
        """返回选课集合的副本，保护封装性"""
        return self._courses.copy()
    
    def add_course(self, course: str) -> None:
        """添加课程"""
        self._courses.add(course)
    
    def remove_course(self, course: str) -> None:
        """移除课程"""
        self._courses.discard(course)  # 使用discard避免KeyError
    
    def clear_courses(self) -> None:
        """清空所有选课（为系统调整新增的方法）"""
        self._courses.clear()
    
    def __hash__(self) -> int:
        """
        哈希方法：只基于学号计算哈希值
        这样两个学号相同但其他属性不同的对象会被视为同一个键
        """
        return hash(self.student_id)
    
    def __eq__(self, other: object) -> bool:
        """
        相等性比较：只比较学号
        确保哈希一致性：如果 x == y，则 hash(x) == hash(y)
        """
        if not isinstance(other, Student):
            return NotImplemented
        return self.student_id == other.student_id
    
    def __lt__(self, other: 'Student') -> bool:
        """小于比较，用于排序"""
        return self.student_id < other.student_id
    
    def __repr__(self) -> str:
        """开发人员友好的表示"""
        courses_str = f", courses={sorted(self._courses)}" if self._courses else ""
        return f"Student('{self.student_id}', '{self.name}'{courses_str})"
    
    def __str__(self) -> str:
        """用户友好的表示"""
        courses_str = ', '.join(sorted(self._courses)) if self._courses else "无选课"
        return f"学生：{self.name}({self.student_id}) 选课：{courses_str}"


# -------------------- 选课系统类 --------------------
class CourseSystem:
    """选课系统，管理课程信息和学生选课"""
    
    def __init__(self):
        # 课程信息：名称 -> 学分
        self.courses_info = {
            '三宝': 2,
            '音数': 2,
            '地概': 2,
            '高数': 4,
            '线代': 3,
            '棒垒': 1
        }
        self.course_capacity = 3  # 每门课人数上限
        self.credit_limit = 6      # 每人学分上限
        self.students = []         # 学生列表
    
    def add_student(self, student: Student):
        """添加学生到系统"""
        self.students.append(student)
    
    def generate_random_students(self, num: int = 8):
        """生成随机学生，每个学生随机选择若干课程作为初始志愿"""
        names = ['张三', '李四', '王五', '赵六', '周七', '吴八', '郑九', '孙十', '陈十一', '刘十二']
        courses_list = list(self.courses_info.keys())
        
        for i in range(num):
            student_id = f"2024{100+i:03d}"
            name = random.choice(names) + str(i)  # 避免重名
            # 随机选择1~4门课（可能超学分，但没关系）
            num_courses = random.randint(1, 4)
            chosen = random.sample(courses_list, num_courses)
            student = Student(student_id, name, set(chosen))
            self.add_student(student)
    
    def print_students(self, title: str):
        """打印所有学生当前选课情况"""
        print(f"\n{title}")
        for s in self.students:
            print(s)
    
    def optimize_enrollment(self):
        """
        静态优化：根据学生当前选课志愿（可超限），重新分配以满足约束
        采用贪心算法：随机打乱学生顺序，按学分降序尝试加入课程
        """
        # 1. 保存每个学生的原始志愿（当前选课）
        original_wishes = {}
        for student in self.students:
            original_wishes[student.student_id] = set(student.courses)  # 复制一份
        
        # 2. 清空所有学生的当前选课
        for student in self.students:
            student.clear_courses()
        
        # 3. 初始化课程当前人数统计
        course_count = {course: 0 for course in self.courses_info}
        # 学生当前已选学分（字典，学生ID -> 学分）
        student_credits = {s.student_id: 0 for s in self.students}
        
        # 4. 随机打乱学生顺序（保证公平）
        shuffled_students = self.students.copy()
        random.shuffle(shuffled_students)
        
        # 5. 按顺序处理每个学生
        for student in shuffled_students:
            # 获取该学生的原始志愿
            wishes = original_wishes[student.student_id]
            if not wishes:
                continue  # 没有志愿，跳过
            
            # 将志愿课程按学分降序排序（优先保留高学分课）
            sorted_courses = sorted(wishes, key=lambda c: self.courses_info[c], reverse=True)
            
            # 尝试加入每门课
            for course in sorted_courses:
                credit = self.courses_info[course]
                # 检查课程是否还有空位，且学生学分未超限
                if course_count[course] < self.course_capacity and \
                   student_credits[student.student_id] + credit <= self.credit_limit:
                    # 可以加入
                    student.add_course(course)
                    course_count[course] += 1
                    student_credits[student.student_id] += credit
        
        # 6. 完成后，所有学生选课已更新
        print("\n=== 优化完成 ===")
    
    def check_constraints(self):
        """检查当前所有学生的选课是否满足约束，返回 (是否满足, 违规信息)"""
        # 检查每人学分
        credit_ok = True
        for s in self.students:
            total = sum(self.courses_info.get(c, 0) for c in s.courses)
            if total > self.credit_limit:
                print(f"违规：{s} 学分 {total} > {self.credit_limit}")
                credit_ok = False
        
        # 检查每课人数
        course_count = {c: 0 for c in self.courses_info}
        for s in self.students:
            for c in s.courses:
                course_count[c] += 1
        capacity_ok = True
        for c, cnt in course_count.items():
            if cnt > self.course_capacity:
                print(f"违规：课程 {c} 人数 {cnt} > {self.course_capacity}")
                capacity_ok = False
        
        return credit_ok and capacity_ok


# -------------------- 测试运行 --------------------
if __name__ == "__main__":
    # 创建选课系统
    system = CourseSystem()
    
    # 生成随机学生（模拟初始选课）
    system.generate_random_students(num=10)  # 10个学生
    
    # 打印初始选课（可能超限）
    system.print_students("=== 初始选课（可超限）===")
    
    # 进行优化分配
    system.optimize_enrollment()
    
    # 打印优化后结果
    system.print_students("=== 优化后最终选课 ===")
    
    # 检查约束
    if system.check_constraints():
        print("\n所有约束满足！")
    else:
        print("\n存在违规！")


=== 初始选课（可超限）===
学生：周七0(2024100) 选课：三宝, 地概, 线代, 音数
学生：刘十二1(2024101) 选课：棒垒, 线代, 音数
学生：郑九2(2024102) 选课：三宝, 线代, 高数
学生：陈十一3(2024103) 选课：三宝, 地概, 线代, 高数
学生：王五4(2024104) 选课：三宝
学生：刘十二5(2024105) 选课：三宝, 棒垒, 音数, 高数
学生：周七6(2024106) 选课：棒垒, 线代
学生：李四7(2024107) 选课：棒垒, 线代
学生：刘十二8(2024108) 选课：三宝, 高数
学生：张三9(2024109) 选课：三宝, 地概

=== 优化完成 ===

=== 优化后最终选课 ===
学生：周七0(2024100) 选课：三宝, 线代
学生：刘十二1(2024101) 选课：棒垒, 音数
学生：郑九2(2024102) 选课：高数
学生：陈十一3(2024103) 选课：三宝, 高数
学生：王五4(2024104) 选课：无选课
学生：刘十二5(2024105) 选课：三宝, 高数
学生：周七6(2024106) 选课：棒垒, 线代
学生：李四7(2024107) 选课：棒垒, 线代
学生：刘十二8(2024108) 选课：无选课
学生：张三9(2024109) 选课：地概

所有约束满足！
